# 📗 Bài 1 — Tokenizer

**Tuần 14**

> Chạy trước: `python curriculum/06-nlp-transformers/tao_du_lieu.py`

Mạng neural chỉ hiểu số. Bài này trả lời câu hỏi: **văn bản biến thành số như thế nào?**

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]

DATA = Path("../../../data")
if not (DATA / "van_ban_viet.txt").exists():
    DATA = Path("data")

van_ban = (DATA / "van_ban_viet.txt").read_text(encoding="utf-8")
print(f"Đã nạp {len(van_ban):,} ký tự")
print(van_ban[:200])

## 1. Tokenizer mức KÝ TỰ — đơn giản nhất

In [ ]:
chars = sorted(set(van_ban))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

print(f"Từ vựng: {len(chars)} ký tự khác nhau")
print("".join(chars[:60]))
print()

ma_hoa = lambda s: [stoi[c] for c in s]
giai_ma = lambda ids: "".join(itos[i] for i in ids)

# Lay mot doan CO THAT trong kho ngu lieu (moi ky tu chac chan co trong tu vung)
mau = van_ban[:16]
print(f"  {mau!r}")
print(f"  -> {ma_hoa(mau)}")
print(f"  -> {giai_ma(ma_hoa(mau))!r}")
print()
print("  Ky tu KHONG co trong tu vung se gay KeyError:")
print(f"  'X' co trong tu vung khong? {'X' in stoi}")
print("  -> Do la ly do tokenizer that luon can token <unk> hoac lam viec tren BYTE.")

> ⚠️ **`sorted()` là bắt buộc.** `set()` không đảm bảo thứ tự ổn định giữa các lần chạy Python. Nếu từ vựng khác nhau giữa lúc train và lúc nạp lại, model sẽ giải mã ra văn bản vô nghĩa hoàn toàn.

**Lưu ý riêng cho tiếng Việt:** chữ `"ế"` có thể là **một** ký tự Unicode (dạng NFC) hoặc **hai** ký tự (dạng NFD: `e` + dấu mũ + dấu sắc). Nhìn giống hệt nhau nhưng từ vựng khác hẳn. Trong dự án thật nên chuẩn hoá trước:

```python
import unicodedata
van_ban = unicodedata.normalize("NFC", van_ban)
```

## 2. Tokenizer mức TỪ

In [ ]:
tu = van_ban.lower().split()
dem_tu = Counter(tu)

print(f"Tổng số từ    : {len(tu):,}")
print(f"Từ khác nhau  : {len(dem_tu):,}")
print()
print("10 từ phổ biến nhất:")
for t, n in dem_tu.most_common(10):
    print(f"  {t:<12} {n:>6,}")

In [ ]:
# Phân bố tần suất từ — quy luật Zipf
tan_suat = sorted(dem_tu.values(), reverse=True)

fig, (a1, a2) = plt.subplots(1, 2)
a1.plot(tan_suat, lw=2, color=MAU_CHINH)
a1.set_xlabel("hạng của từ"); a1.set_ylabel("tần suất")
a1.set_title("Phân bố tần suất từ")

a2.loglog(tan_suat, lw=2, color=MAU_CHINH)
a2.set_xlabel("hạng (log)"); a2.set_ylabel("tần suất (log)")
a2.set_title("Cùng dữ liệu, thang log-log")
plt.tight_layout(); plt.show()

print(f"  {sum(1 for n in dem_tu.values() if n == 1)} từ chỉ xuất hiện ĐÚNG MỘT LẦN")

**Quy luật Zipf:** vài từ xuất hiện cực nhiều, còn *rất nhiều* từ chỉ xuất hiện một hai lần. Trên thang log-log, đường này gần như thẳng — đúng với **mọi** ngôn ngữ tự nhiên.

**Hệ quả cho tokenizer mức từ:**
- Muốn phủ hết từ → từ vựng khổng lồ, phần lớn là từ hiếm model không học nổi
- Cắt bớt từ vựng → mọi từ hiếm thành `<unk>`, mất sạch thông tin

Với tiếng Việt còn tệ hơn: `"học"`, `"học tập"`, `"đi học"`, `"học sinh"` là **bốn** token hoàn toàn khác nhau, dù chúng chia sẻ chữ `"học"`.

> Đó chính là vấn đề mà **BPE** giải quyết.

## 3. ⭐ BPE — thuật toán GPT dùng

In [ ]:
def dem_cap(toks):
    return Counter(zip(toks, toks[1:]))


def gop_cap(toks, cap):
    ket_qua, i = [], 0
    while i < len(toks):
        if i < len(toks) - 1 and (toks[i], toks[i + 1]) == cap:
            ket_qua.append(toks[i] + toks[i + 1]); i += 2
        else:
            ket_qua.append(toks[i]); i += 1
    return ket_qua


# Chạy từng bước trên ví dụ nhỏ
demo = "nam nam nam nha nha"
toks = list(demo)
print(f"  bước 0: {toks}")

for buoc in range(1, 5):
    d = dem_cap(toks)
    if not d:
        break
    cao_nhat = max(d.values())
    cap = min(c for c, n in d.items() if n == cao_nhat)
    toks = gop_cap(toks, cap)
    print(f"  bước {buoc}: gộp {cap} (xuất hiện {cao_nhat} lần)  ->  {toks}")

```
① Bắt đầu: mỗi ký tự là một token
② Tìm CẶP token liền nhau xuất hiện NHIỀU NHẤT
③ Gộp cặp đó thành một token mới
④ Lặp lại N lần
```

> Ý tưởng ban đầu không đến từ NLP mà từ **nén dữ liệu** (Gage, 1994): thay cặp byte hay gặp nhất bằng một byte chưa dùng. Sennrich (2015) mang nó sang dịch máy và nó trở thành chuẩn của cả ngành.

In [ ]:
def huan_luyen_bpe(vb, so_lan):
    toks, gop = list(vb), []
    for _ in range(so_lan):
        d = dem_cap(toks)
        if not d:
            break
        cao_nhat = max(d.values())
        cap = min(c for c, n in d.items() if n == cao_nhat)
        gop.append(cap)
        toks = gop_cap(toks, cap)
    return gop


def ap_dung_bpe(vb, gop):
    toks = list(vb)
    for cap in gop:
        toks = gop_cap(toks, cap)
    return toks


mau_vb = van_ban[:20000]
phep_gop = huan_luyen_bpe(mau_vb, 200)

print("20 phép gộp đầu tiên BPE học được:")
print("  " + "  ".join(f"{a!r}+{b!r}" for a, b in phep_gop[:20]))
print()
print("Vài token dài nhất sau 200 phép gộp:")
tokens = sorted(set(ap_dung_bpe(mau_vb, phep_gop)), key=len, reverse=True)
print("  " + "  ".join(repr(t) for t in tokens[:15]))

> 🔍 Nhìn kỹ danh sách token dài nhất: BPE **tự học** ra các từ và cụm từ tiếng Việt hay lặp lại — không ai dạy nó khái niệm 'từ' cả. Nó chỉ đếm cặp ký tự.

## 4. ⭐ So sánh ba mức — đánh đổi cốt lõi

In [ ]:
tv_ky_tu = sorted(set(mau_vb))
ma_ky_tu = [c for c in mau_vb]

tu_mau = mau_vb.lower().split()
tv_tu = set(tu_mau)

ma_bpe = ap_dung_bpe(mau_vb, phep_gop)

bang = [
    ("ký tự", len(ma_ky_tu), len(tv_ky_tu)),
    ("BPE (200 gộp)", len(ma_bpe), len(set(ma_bpe))),
    ("từ", len(tu_mau), len(tv_tu)),
]

print(f"  {'mức':<16} {'số token':>10} {'từ vựng':>10}")
print("  " + "-" * 38)
for ten, n_tok, n_tv in bang:
    print(f"  {ten:<16} {n_tok:>10,} {n_tv:>10,}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2)
ten = [b[0] for b in bang]

a1.barh(ten[::-1], [b[1] for b in bang][::-1], color=MAU_CHINH, height=.55)
a1.set_xlabel("số token"); a1.set_title("Độ dài chuỗi (càng ngắn càng rẻ)")
a1.grid(axis="y", visible=False)

a2.barh(ten[::-1], [b[2] for b in bang][::-1], color=MAU[2], height=.55)
a2.set_xlabel("kích thước từ vựng"); a2.set_title("Từ vựng (càng nhỏ càng nhẹ)")
a2.grid(axis="y", visible=False)

plt.tight_layout(); plt.show()

```
        từ vựng nhỏ                          từ vựng lớn
        chuỗi dài                            chuỗi ngắn
        ├──────────────┼──────────────┼──────────────┤
      ký tự         subword          từ
                   ← LLM ở đây →
```

**Vì sao độ dài chuỗi quan trọng đến vậy?** Attention có độ phức tạp **O(n²)**. Chuỗi dài gấp đôi → tính toán gấp **bốn**.

Với đoạn văn 20.000 ký tự ở trên:
- mức ký tự: 20.000 token → attention tính ~400 triệu phép
- mức BPE: ~10.600 token → ~112 triệu phép

Nhanh hơn gần 4 lần chỉ nhờ đổi tokenizer.

**Vì sao không dùng từ vựng thật lớn để chuỗi thật ngắn?** Vì ma trận embedding có kích thước `(vocab × d_model)`. Với `d=4096` và vocab 1 triệu → **4 tỷ tham số** chỉ cho embedding. Khoảng 30.000–130.000 là vùng cân bằng mà cả ngành đã hội tụ về.

## 5. ⚠️ Ba điều về token mọi AI Engineer phải biết

In [ ]:
cap_song_ngu = [
    ("Xin chào, bạn khoẻ không?", "Hello, how are you?"),
    ("Tôi đang học lập trình Python.", "I am learning Python programming."),
    ("Hôm nay trời mưa rất to.", "It is raining heavily today."),
]

print(f"  {'tiếng Việt':<34} {'ký tự':>7} | {'tiếng Anh':<32} {'ký tự':>7}")
print("  " + "-" * 92)
for vi, en in cap_song_ngu:
    print(f"  {vi:<34} {len(vi):>7} | {en:<32} {len(en):>7}")

print()
print("  Đó mới là KÝ TỰ. Với tokenizer thật (huấn luyện chủ yếu trên tiếng Anh),")
print("  tiếng Việt còn bị tách vụn hơn nữa vì các ký tự có dấu ít gặp.")

**① Tiếng Việt tốn token hơn tiếng Anh.** Tokenizer của các LLM lớn được huấn luyện chủ yếu trên tiếng Anh, nên chữ có dấu thường bị tách vụn. **Cùng một ý, prompt tiếng Việt tốn nhiều token hơn — nghĩa là tốn tiền hơn.**

**② Đừng ước lượng chi phí bằng số từ.** Tỷ lệ "1 token ≈ 0.75 từ" chỉ đúng với tiếng Anh. Với tiếng Việt phải **đếm token thật** (Phase 07).

**③ Model không "thấy" chữ cái — nó thấy token.**

Đó là lý do LLM hay sai khi được hỏi *"từ 'dâu tây' có mấy chữ â?"* hay *"đảo ngược chuỗi này"*. Nó chưa bao giờ nhìn thấy từng ký tự riêng lẻ — nó chỉ thấy các khối token.

> 💡 Nếu cần LLM xử lý ở mức ký tự, hãy **chèn dấu cách giữa các chữ cái** trong prompt để ép tokenizer tách nhỏ ra.

## 6. Token đặc biệt

Mọi tokenizer thật đều có các token đặc biệt:

| Token | Vai trò |
|---|---|
| `<unk>` | Từ không biết |
| `<pad>` | Đệm cho đủ độ dài batch |
| `<bos>` / `<eos>` | Bắt đầu / kết thúc chuỗi |
| `<sep>` | Ngăn cách hai đoạn |

**`<eos>` đặc biệt quan trọng:** đó là cách LLM biết **khi nào dừng sinh**. Khi model sinh ra token `<eos>`, vòng lặp sinh kết thúc.

> 🔗 Trong Claude/GPT, các token đặc biệt đánh dấu vai trò (`system` / `user` / `assistant`) chính là cơ chế kỹ thuật đứng sau khái niệm **chat template** mà bạn sẽ dùng ở Phase 07.

## ✍️ Bài tập

```bash
pytest tests/phase06/test_ex01.py -v
```